In [7]:
import os
import shutil
import time

BASE_DIR = "/content/integracao-cotacoes"
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)

files = {}

files["app/__init__.py"] = "# Pacote app\n"

files["app/exceptions.py"] = (
    "class ErroComunicacaoAPI(Exception):\n    pass\n\n"
    "class RespostaInvalidaAPI(Exception):\n    pass\n"
)

files["app/api_client.py"] = (
    "import os\nimport requests\nimport time\n"
    "from app.exceptions import ErroComunicacaoAPI, RespostaInvalidaAPI\n\n"
    "class APIClient:\n"
    "    def __init__(self, base_url='https://economia.awesomeapi.com.br', timeout=5, tentativas=3, espera=1.0, api_key=None):\n"
    "        self.base_url = base_url\n"
    "        self.timeout = timeout\n"
    "        self.tentativas = tentativas\n"
    "        self.espera = espera\n"
    "        self.api_key = api_key or os.environ.get('AWESOMEAPI_KEY')  # chave opcional\n\n"
    "    def buscar_cotacoes(self, pares='USD-BRL,EUR-BRL,BTC-BRL'):\n"
    "        url = f'{self.base_url}/json/last/{pares}'\n"
    "        headers = {'x-api-key': self.api_key} if self.api_key else {}\n"
    "        ultimo_erro = None\n"
    "        for tentativa in range(1, self.tentativas + 1):\n"
    "            pausa = self.espera * tentativa  # espera crescente\n"
    "            try:\n"
    "                response = requests.get(url, headers=headers, timeout=self.timeout)\n"
    "                if response.status_code == 429:  # limite de requisicoes: tenta de novo\n"
    "                    pausa = self._pausa_429(response, pausa)\n"
    "                    dica = '' if self.api_key else ' - sem chave de API; defina AWESOMEAPI_KEY'\n"
    "                    raise ErroComunicacaoAPI('Limite de requisicoes da API atingido (HTTP 429)' + dica)\n"
    "                if response.status_code >= 500:\n"
    "                    raise ErroComunicacaoAPI(f'API indisponivel (HTTP {response.status_code})')\n"
    "                if response.status_code != 200:  # demais 4xx: repetir nao resolve\n"
    "                    raise RespostaInvalidaAPI(f'Requisicao rejeitada (HTTP {response.status_code})')\n"
    "                try:\n"
    "                    return response.json()\n"
    "                except ValueError as exc:\n"
    "                    raise RespostaInvalidaAPI('A resposta nao e um JSON valido') from exc\n"
    "            except RespostaInvalidaAPI:\n"
    "                raise\n"
    "            except (requests.Timeout, requests.ConnectionError) as exc:\n"
    "                ultimo_erro = ErroComunicacaoAPI(f'Falha de comunicacao: {exc}')\n"
    "            except ErroComunicacaoAPI as exc:\n"
    "                ultimo_erro = exc\n"
    "            if tentativa < self.tentativas:\n"
    "                time.sleep(pausa)\n"
    "        raise ultimo_erro\n\n"
    "    def _pausa_429(self, response, padrao):\n"
    "        try:\n"
    "            pedida = float(response.headers.get('Retry-After', ''))\n"
    "        except ValueError:\n"
    "            return padrao * 2\n"
    "        return min(max(pedida, 0), 30)\n"
)

files["app/transformer.py"] = (
    "from datetime import datetime, timezone\nfrom app.exceptions import RespostaInvalidaAPI\n\n"
    "class Transformer:\n"
    "    @staticmethod\n"
    "    def transformar(dados_json):\n"
    "        if not isinstance(dados_json, dict) or not dados_json:\n"
    "            raise RespostaInvalidaAPI('Resposta invalida: esperava um dicionario JSON nao vazio')\n"
    "        registos = []\n"
    "        for chave, item in dados_json.items():\n"
    "            try:\n"
    "                compra, venda = float(item['bid']), float(item['ask'])\n"
    "                cotado_em = datetime.fromtimestamp(int(item['timestamp']), tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')\n"
    "                registos.append({\n"
    "                    'par': f\"{item['code']}-{item['codein']}\",\n"
    "                    'compra': compra, 'venda': venda, 'spread': round(venda - compra, 4),\n"
    "                    'variacao': float(item['pctChange']), 'cotado_em': cotado_em,\n"
    "                    'criado_em': datetime.now(timezone.utc).isoformat()\n"
    "                })\n"
    "            except Exception as e:\n"
    "                raise RespostaInvalidaAPI(f'Erro ao processar {chave}: {e}')\n"
    "        return registos\n"
)

files["app/repository.py"] = (
    "import sqlite3\n\n"
    "class Repository:\n"
    "    def __init__(self, db_path='cotacoes.db'):\n"
    "        self.db_path = db_path\n"
    "        self._criar_tabela()\n\n"
    "    def _criar_tabela(self):\n"
    "        with sqlite3.connect(self.db_path) as conn:\n"
    "            conn.execute('''CREATE TABLE IF NOT EXISTS cotacoes (\n"
    "                id INTEGER PRIMARY KEY AUTOINCREMENT, par TEXT NOT NULL, compra REAL,\n"
    "                venda REAL, spread REAL, variacao REAL, cotado_em TEXT, criado_em TEXT,\n"
    "                UNIQUE(par, cotado_em))''')\n\n"
    "    def salvar_cotacoes(self, cotacoes):\n"
    "        with sqlite3.connect(self.db_path) as conn:\n"
    "            query = 'INSERT OR IGNORE INTO cotacoes (par, compra, venda, spread, variacao, cotado_em, criado_em) VALUES (?, ?, ?, ?, ?, ?, ?)'\n"
    "            dados = [(c['par'], c['compra'], c['venda'], c['spread'], c['variacao'], c['cotado_em'], c['criado_em']) for c in cotacoes]\n"
    "            conn.executemany(query, dados)\n\n"
    "    def listar_todas(self, par=None):\n"
    "        with sqlite3.connect(self.db_path) as conn:\n"
    "            if par: return conn.execute('SELECT * FROM cotacoes WHERE par = ? ORDER BY cotado_em DESC', (par,)).fetchall()\n"
    "            return conn.execute('SELECT * FROM cotacoes ORDER BY cotado_em DESC').fetchall()\n"
)

files["app/service.py"] = (
    "from app.api_client import APIClient\nfrom app.transformer import Transformer\nfrom app.repository import Repository\n\n"
    "class CotacaoService:\n"
    "    def __init__(self, db_path='cotacoes.db', client=None):\n"
    "        self.client = client or APIClient()\n"
    "        self.transformer = Transformer()\n"
    "        self.repo = Repository(db_path=db_path)\n\n"
    "    def sincronizar(self, pares='USD-BRL,EUR-BRL,BTC-BRL'):\n"
    "        registos = self.transformer.transformar(self.client.buscar_cotacoes(pares))\n"
    "        self.repo.salvar_cotacoes(registos)\n"
    "        return len(registos)\n\n"
    "    def listar(self, par=None): return self.repo.listar_todas(par)\n"
)

files["app/main.py"] = (
    "import sys\nfrom app.service import CotacaoService\nfrom app.exceptions import ErroComunicacaoAPI, RespostaInvalidaAPI\n\n"
    "def main():\n"
    "    service = CotacaoService()\n"
    "    args = sys.argv[1:]\n"
    "    if not args or args[0] == 'sync':\n"
    "        try:\n"
    "            total = service.sincronizar()\n"
    "        except (ErroComunicacaoAPI, RespostaInvalidaAPI) as e:\n"
    "            print(f'\\n Erro na integração: {e}\\n')\n"
    "            sys.exit(1)\n"
    "        print(f'\\n Sincronização concluída! {total} cotações salvas no banco.\\n')\n"
    "    elif args[0] == 'listar':\n"
    "        registos = service.listar()\n"
    "        print(f'\\n Total no banco: {len(registos)}\\n' + '-'*90)\n"
    "        print(f\"{'ID':<4} | {'PAR':<8} | {'COMPRA':<11} | {'VENDA':<11} | {'SPREAD':<8} | {'VARIAÇÃO':<9} | {'DATA'}\")\n"
    "        print('-'*90)\n"
    "        for r in registos:\n"
    "            print(f\"{r[0]:<4} | {r[1]:<8} | R$ {r[2]:<8.2f} | R$ {r[3]:<8.2f} | R$ {r[4]:<5.2f} | {r[5]:>8.2f}% | {r[6]}\")\n"
    "        print('-'*90 + '\\n')\n"
    "if __name__ == '__main__': main()\n"
)

files["tests/test_integracao.py"] = (
    "import pytest\nimport requests\n"
    "from app.service import CotacaoService\n"
    "from app.api_client import APIClient\n"
    "from app.exceptions import ErroComunicacaoAPI, RespostaInvalidaAPI\n\n"
    "JSON_OK = {'USDBRL': {'code': 'USD', 'codein': 'BRL', 'bid': '5.15', 'ask': '5.17', 'pctChange': '0.45', 'timestamp': '1767225600'}}\n\n"
    "class RespostaFalsa:\n"
    "    def __init__(self, dados=None, status=200, json_valido=True, headers=None):\n"
    "        self.dados, self.status_code, self.json_valido, self.headers = dados, status, json_valido, headers or {}\n"
    "    def json(self):\n"
    "        if not self.json_valido:\n"
    "            raise ValueError('nao e JSON')\n"
    "        return self.dados\n\n"
    "def simular_api(monkeypatch, *respostas):\n"
    "    # Cada chamada devolve (ou levanta) o proximo item; o ultimo se repete.\n"
    "    fila, chamadas = list(respostas), {'n': 0, 'headers': {}}\n"
    "    def get_falso(url, timeout=None, headers=None):\n"
    "        chamadas['n'] += 1\n"
    "        chamadas['headers'] = headers or {}\n"
    "        item = fila.pop(0) if len(fila) > 1 else fila[0]\n"
    "        if isinstance(item, Exception):\n"
    "            raise item\n"
    "        return item\n"
    "    monkeypatch.setattr(requests, 'get', get_falso)\n"
    "    monkeypatch.setattr('app.api_client.time.sleep', lambda s: None)\n"
    "    return chamadas\n\n"
    "def criar_service(tmp_path):\n"
    "    return CotacaoService(db_path=str(tmp_path / 'teste.db'), client=APIClient(tentativas=3))\n\n"
    "def test_fluxo_integracao_completo(tmp_path, monkeypatch):\n"
    "    simular_api(monkeypatch, RespostaFalsa(JSON_OK))\n"
    "    service = criar_service(tmp_path)\n"
    "    assert service.sincronizar('USD-BRL') == 1\n"
    "    linha = service.listar('USD-BRL')[0]\n"
    "    assert linha[1] == 'USD-BRL' and linha[2] == 5.15 and linha[3] == 5.17\n"
    "    assert linha[4] == pytest.approx(0.02)\n"
    "    assert linha[6] == '2026-01-01 00:00:00'\n\n"
    "def test_sincronizar_duas_vezes_nao_duplica(tmp_path, monkeypatch):\n"
    "    simular_api(monkeypatch, RespostaFalsa(JSON_OK))\n"
    "    service = criar_service(tmp_path)\n"
    "    service.sincronizar('USD-BRL'); service.sincronizar('USD-BRL')\n"
    "    assert len(service.listar()) == 1\n\n"
    "def test_timeout_gera_erro_de_comunicacao_e_nao_grava(tmp_path, monkeypatch):\n"
    "    chamadas = simular_api(monkeypatch, requests.Timeout('timeout'))\n"
    "    service = criar_service(tmp_path)\n"
    "    with pytest.raises(ErroComunicacaoAPI):\n"
    "        service.sincronizar('USD-BRL')\n"
    "    assert chamadas['n'] == 3 and service.listar() == []\n\n"
    "def test_http_503_faz_retentativas(tmp_path, monkeypatch):\n"
    "    chamadas = simular_api(monkeypatch, RespostaFalsa({}, status=503))\n"
    "    with pytest.raises(ErroComunicacaoAPI):\n"
    "        criar_service(tmp_path).sincronizar('USD-BRL')\n"
    "    assert chamadas['n'] == 3\n\n"
    "def test_429_tenta_de_novo_e_recupera(tmp_path, monkeypatch):\n"
    "    chamadas = simular_api(monkeypatch, RespostaFalsa({}, status=429, headers={'Retry-After': '0'}), RespostaFalsa(JSON_OK))\n"
    "    assert criar_service(tmp_path).sincronizar('USD-BRL') == 1\n"
    "    assert chamadas['n'] == 2\n\n"
    "def test_http_404_nao_repete(tmp_path, monkeypatch):\n"
    "    chamadas = simular_api(monkeypatch, RespostaFalsa({}, status=404))\n"
    "    with pytest.raises(RespostaInvalidaAPI):\n"
    "        criar_service(tmp_path).sincronizar('USD-BRL')\n"
    "    assert chamadas['n'] == 1\n\n"
    "def test_chave_da_api_e_enviada_no_cabecalho(tmp_path, monkeypatch):\n"
    "    chamadas = simular_api(monkeypatch, RespostaFalsa(JSON_OK))\n"
    "    service = CotacaoService(db_path=str(tmp_path / 'teste.db'), client=APIClient(api_key='minha-chave'))\n"
    "    assert service.sincronizar('USD-BRL') == 1\n"
    "    assert chamadas['headers'] == {'x-api-key': 'minha-chave'}\n\n"
    "def test_json_invalido_e_rejeitado(tmp_path, monkeypatch):\n"
    "    simular_api(monkeypatch, RespostaFalsa(json_valido=False))\n"
    "    service = criar_service(tmp_path)\n"
    "    with pytest.raises(RespostaInvalidaAPI):\n"
    "        service.sincronizar('USD-BRL')\n"
    "    assert service.listar() == []\n\n"
    "def test_campos_faltando_e_rejeitado(tmp_path, monkeypatch):\n"
    "    simular_api(monkeypatch, RespostaFalsa({'USDBRL': {'code': 'USD', 'codein': 'BRL'}}))\n"
    "    service = criar_service(tmp_path)\n"
    "    with pytest.raises(RespostaInvalidaAPI):\n"
    "        service.sincronizar('USD-BRL')\n"
    "    assert service.listar() == []\n"
)

files["requirements.txt"] = "requests>=2.31\npytest>=8.0\n"

files["README.md"] = (
    "# Integração de Cotações de Moedas (AwesomeAPI -> SQLite)\n\n"
    "**Grupo 1:** Ana Luiza, Dieli, Helena e Pedro Alves\n\n"
    "## 1. O que a aplicação faz\nConsome a API pública **AwesomeAPI**, transforma os dados e os grava no **SQLite** local.\n\n"
    "## 2. Arquitetura\n`AwesomeAPI ──HTTP──> api_client ──> transformer ──> repository ──> SQLite`\n\n"
    "## 3. Tecnologias\nPython 3, `requests` (API), `sqlite3` (Banco), `pytest` (Testes).\n\n"
    "## 4. Como executar\n"
    "```bash\n"
    "pip install -r requirements.txt\n"
    "export AWESOMEAPI_KEY=sua_chave   # opcional (veja a seção 5)\n"
    "python -m app.main sync      # busca na API e grava no SQLite (cotacoes.db)\n"
    "python -m app.main listar    # mostra o que foi gravado\n"
    "python -m pytest -v          # roda os testes\n"
    "```\n\n"
    "## 5. Tratamento de falhas e decisões de implementação\n"
    "- **API isolada:** só `app/api_client.py` conhece URL, HTTP, timeout e retry; o restante recebe dados já obtidos.\n"
    "- **Falhas de comunicação:** timeout de 5 s e até 3 tentativas com espera crescente (timeout, conexão, HTTP 5xx). Falha final gera `ErroComunicacaoAPI`.\n"
    "- **HTTP 429 (limite de requisições):** nova tentativa respeitando `Retry-After` (máx. 30 s) ou o dobro da espera normal.\n"
    "- **Respostas inválidas:** HTTP 4xx (exceto 429), JSON inválido, resposta vazia ou campos ausentes/não numéricos geram `RespostaInvalidaAPI`. Nada é gravado nesses casos.\n"
    "- **Chave de API (opcional):** lida da variável de ambiente `AWESOMEAPI_KEY` e enviada no cabeçalho `x-api-key`. A AwesomeAPI oferece chave gratuita; sem ela, a API pode limitar (HTTP 429) em IPs compartilhados, como o do Google Colab.\n"
    "- **Sem dados falsos:** a aplicação nunca grava dados simulados; se a API falha, o erro é mostrado.\n"
    "- **SQLite:** não exige servidor. `UNIQUE(par, cotado_em)` + `INSERT OR IGNORE` evita duplicar a mesma cotação.\n"
    "- **Transformação:** textos viram número, timestamp Unix vira data (UTC) e é calculado o spread (venda - compra).\n"
    "- **Client injetável:** `CotacaoService` recebe o client, o que permite testar sem internet.\n\n"
    "## 6. Testes (9)\n"
    "O teste de integração exercita service, transformer e SQLite reais, com a resposta HTTP simulada: fluxo completo; sem duplicar ao sincronizar duas vezes; timeout; HTTP 503 com 3 tentativas; HTTP 429 com recuperação; HTTP 404 sem repetir; envio da chave no cabeçalho; JSON inválido; campos faltando.\n\n"
    "## 7. Classificação da Integração\n"
    "| Critério | Classificação | Justificativa |\n|---|---|---|\n"
    "| Contexto de negócio | B2B (consumo de serviço de terceiros) | A aplicação consome dados de um serviço mantido por outra organização. |\n"
    "| Padrão arquitetural | Ponto a ponto (Point-to-Point) | Conexão direta entre dois sistemas, sem broker, ESB ou hub. |\n"
    "| Padrão de comunicação | Síncrona, requisição-resposta, REST/JSON via HTTP (pull) | A aplicação pede, espera a resposta e então processa; a iniciativa é nossa, sem eventos. |\n"
)

for rel_path, content in files.items():
    full_path = os.path.join(BASE_DIR, rel_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, "w", encoding="utf-8") as f: f.write(content)

In [8]:
import os
from getpass import getpass

# Opcional: chave gratuita da AwesomeAPI (https://docs.awesomeapi.com.br/instrucoes-api-key).
# Ajuda a evitar o erro 429 (limite de requisições) no Colab. Pode deixar em branco (Enter).
chave = getpass('AWESOMEAPI_KEY (opcional): ')
if chave:
    os.environ['AWESOMEAPI_KEY'] = chave
    print(' Chave configurada.')
else:
    print('Sem chave: a API pode responder 429 em horários de muito uso.')


AWESOMEAPI_KEY (opcional): ··········
 Chave configurada.


In [9]:
%cd /content/integracao-cotacoes

# 1. Instala as dependências necessárias
!pip install -q -r requirements.txt

# 2. Busca os dados na AwesomeAPI e salva no banco de dados
!python -m app.main sync

# 3. Lista a tabela formatada com os valores (Com data correta e R$)
!python -m app.main listar

# 4. Roda a automação de testes validando a integração
!python -m pytest -v

/content/integracao-cotacoes

 Sincronização concluída! 3 cotações salvas no banco.


 Total no banco: 3
------------------------------------------------------------------------------------------
ID   | PAR      | COMPRA      | VENDA       | SPREAD   | VARIAÇÃO  | DATA
------------------------------------------------------------------------------------------
3    | BTC-BRL  | R$ 432015.00 | R$ 432072.00 | R$ 57.00 |     1.99% | 2026-11-04 11:32:11
2    | EUR-BRL  | R$ 6.17     | R$ 6.18     | R$ 0.00  |     0.40% | 2026-11-04 11:29:28
1    | USD-BRL  | R$ 5.49     | R$ 5.49     | R$ 0.00  |     0.00% | 2026-11-04 07:33:03
------------------------------------------------------------------------------------------

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/integracao-cotacoes
plugins: langsmith-0.14.15, typeguard-4.6.0, platformd

**restaurar pasta do projeto (Ana)**

rode este código para restaurar a pasta do projeto a partir do Drive

In [ ]:
from google.colab import drive
import shutil

# Conecta ao Drive
drive.mount('/content/drive')

# Copia do Drive de volta para o ambiente do Colab
shutil.copytree('/content/drive/MyDrive/integracao-cotacoes', '/content/integracao-cotacoes', dirs_exist_ok=True)

print("✅ Projeto restaurado do Drive! Você já pode rodar 'python -m app.main listar'.")